# Pier's distribution experiments

What changes with width: the entire distribution of minibatch gradient inner products,
its skewness and tails, alignment with the exact full-training gradient, and alignment
with the actual step and momentum buffer. Both layers and hidden biases train.

The main study uses widths **8, 9, 10, 16, 32**, ten consecutive seeds, and 100k updates.
An earlier loss-matched comparison uses forty seeds. All use the same eight-neuron
teacher, Gaussian inputs, batch size 64, and 128 fixed probe batches per snapshot.
The 8,128 pairwise products within a snapshot are dependent; intervals resample seeds.

Run `scripts/analyze_distributions.py` to regenerate figures/tables. This notebook only
reads data; it never starts training. See `docs/pier_experiments.md` for interpretation.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == 'analyses':
    ROOT = ROOT.parent
OUT = ROOT / 'runs/pier_study/analysis'
assert (OUT / 'early_statistics.csv').exists(), 'Run scripts/analyze_distributions.py first'

def figure(name):
    display(Image(filename=str(OUT / f'{name}.png')))

early = pd.read_csv(OUT / 'early_statistics.csv')
matched = pd.read_csv(OUT / 'matched_statistics.csv')
outcomes = pd.read_csv(OUT / 'outcomes.csv')
display(early.groupby(['optimizer','width']).agg(
    seeds=('seed','nunique'), mse_mean=('train_mse','mean'),
    mse_min=('train_mse','min'), mse_max=('train_mse','max')))


## Full distributions at comparable early loss

Here all three optimizers can be compared near ordinary training MSE 0.2.
The top row keeps the magnitude of the raw inner product. The bottom divides
by the mean squared minibatch-gradient norm at that snapshot; this separates
scale changes from changes in relative geometry. Each seed contributes equally.
Shading is a pointwise 95% seed-bootstrap interval, not a simultaneous confidence band.

In [ ]:
figure('gradient_distributions_0.2')
figure('gradient_tails_0.2')

## Width effects and uncertainty

Skewness is the standardized third central moment; excess kurtosis is the standardized
fourth moment minus three. These describe empirical tail/asymmetry structure. They do
not establish a power-law tail or a particular asymptotic distribution. The paired
width estimates compare the same training seed and minibatch sequence at two widths.
Intervals are exploratory and have no multiple-comparison correction.

In [ ]:
figure('width_statistics_0.2')
effects = pd.read_csv(OUT / 'early_paired_effects.csv')
metrics = ['scaled_dot_skewness','scaled_dot_excess_kurtosis',
           'cosine_negative_fraction','step_alignment_negative_fraction']
display(effects[(effects.large == 16) & effects.metric.isin(metrics)])

### Fresh tail probes

The main plots condition on one fixed set of 128 probe batches. To check sampling
sensitivity, widths 8 and 16 are remeasured at the saved early states using 512 new
batches, with all forty training seeds and no retraining. This gives 130,816 distinct
pairs per state, still dependent. Fourth-moment estimates can move substantially
with the probe set. Compare qualitative tail differences and report this sensitivity.

In [ ]:
display(pd.read_csv(OUT / 'tail_validation_summary.csv'))
display(pd.read_csv(OUT / 'tail_validation_effects.csv'))

## Gradients, moments, and actual steps

A negative gradient-pair dot predicts conflict for an infinitesimal SGD step on that
other batch. Momentum and Adam use a different direction. Here the updated moment and
actual descent direction are projected onto the same independent probe gradients.
Positive alignment with the descent direction predicts improvement. Adam's raw first
moment and its preconditioned update can also differ substantially.

These are distributions of projections of one saved buffer/step, not independent
optimizer histories sampled at a fixed parameter state.

In [ ]:
figure('gradient_moment_step_distributions')

## Later, near loss 0.01

This comparison is restricted to runs with an observed training MSE within 25% of
0.01. Some Adam/Muon runs never reach this level. Count them as failures to reach the
comparison, not as missing successful observations. Do not infer an optimizer effect
by comparing this selected subset with the complete early comparison.

In [ ]:
eligibility = pd.read_csv(OUT / 'loss_matching.csv')
display(eligibility.groupby(['collection','optimizer','width']).agg(
    available=('seed','count'), eligible=('loss_eligible','sum')))
figure('gradient_distributions_0.01')
figure('width_statistics_0.01')

## Training outcomes and initialization control

All errors below are ordinary MSE. Main-study success uses test MSE < 1e-6 after 100k
updates. The learning-rate/Muon pilot has a shorter 20k budget and five seeds, so it
must be interpreted separately. Muon operates on the hidden weight matrix, with
Adam on biases/output coefficients and no weight decay.

The default outgoing coefficients are exactly zero. The separate Adam control changes
only their initialization to N(0, 0.1²), leaving the incoming weights, data, minibatch
sequence and training budget paired. It tests initialization sensitivity, not a fair
ranking of optimally tuned optimizers.

In [ ]:
figure('training')
display(pd.read_csv(OUT / 'outcomes_summary.csv'))
if (OUT / 'initialization_summary.csv').exists():
    display(pd.read_csv(OUT / 'initialization_summary.csv'))
if (OUT / 'sign_diagnostics.csv').exists():
    signs = pd.read_csv(OUT / 'sign_diagnostics.csv')
    display(signs.groupby(['collection','optimizer','width']).agg(
        seeds=('seed','count'), positive_units_mean=('positive_units','mean'),
        median_test_mse=('test_mse','median')))


## Curvature, oscillations and finite steps

The paper's batch sharpness is E[gBᵀ HB gB] / E[||gB||²]. The old code's mean of
individual ratios is retained under the old name, but is a different statistic.
The corrected field is `batch_sharpness_ratio` and the raw numerators/denominators
are saved. This curvature is for the half-MSE objective.

A scalar plateau alone does not establish EoSS. SGD's 2/lr threshold cannot simply
be applied to Adam. Momentum has a batch-size-dependent stability regime.
The high-rate pilot and exact one-step rescaling compare stability/finite-step
behavior directly. The quadratic local model excludes ReLU gate crossings; the
exact float64 loss evaluation includes them.

In [ ]:
figure('evolution_sgd_1')
figure('evolution_momentum_01')
figure('evolution_adam_0001')
figure('finite_step_intervention')
if (OUT / 'stability_pilot.png').exists():
    figure('stability_pilot')
if (OUT / 'stability_summary.csv').exists():
    display(pd.read_csv(OUT / 'stability_summary.csv'))


### Saved-state learning-rate intervention

All four continuations start from each seed's **identical** width-16 parameters after
20,000 SGD updates at learning rate 4. They receive the same subsequent minibatches;
plain SGD has no momentum state to restore. Only learning rate changes. This directly
tests the cause of that particular high-rate plateau; it does not intervene on the
gradient-distribution statistic itself.

In [ ]:
figure('stability_continuation')
display(pd.read_csv(OUT / 'continuation_summary.csv'))
display(pd.read_csv(OUT / 'continuation_paired_effects.csv'))

## What this can establish

The measurements diagnose optimizer-dependent geometry and finite-step behavior on
one fixed teacher. A correlation between a distribution statistic and eventual fit
is not a demonstration that the statistic causes the width benefit. The sampled
widths also cannot establish a sharp phase transition. Layer normalization, attention,
CIFAR and language are separate follow-up experiments, not part of this shallow-ReLU
study. Exact numerical findings and next steps are in `docs/pier_experiments.md`.